# Data preparation and patient separation

**Purpose:** Verify source integrity, eligibility, target definitions and patient-disjoint partitions before any modeling.

Requires the complete project folder and dependencies from `requirements.txt`; see README.md for local/Colab/Kaggle setup. Matching saved artifacts are reused. Use `force=True` in the phase cell to recompute. Outputs must remain visible when saving.

In [1]:
from pathlib import Path
import os, sys
for name in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ.setdefault(name, '2')
candidates = [Path.cwd(), *Path.cwd().parents,
              Path('/content/hospital-readmission'), Path('/kaggle/working/hospital-readmission')]
ROOT = next((p for p in candidates if (p / 'config.json').exists() and (p / 'readmission').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Extract the full project and set the working directory to hospital-readmission first. See README.md.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image, JSON
from readmission.common import read_json
from readmission.pipeline import run_phase
pd.set_option('display.max_columns', 30)
print('Project:', ROOT.name)


Project: hospital-readmission


In [2]:
run_phase('data')

Phase data: reusing verified matching artifacts.


In [3]:
display(JSON(read_json('artifacts/data_audit.json')))

<IPython.core.display.JSON object>

In [4]:
display(pd.read_csv(ROOT / 'artifacts/tables/split_summary.csv'))

,split,encounters,patients,positives,positive_rate
0,calibration,19746,14025,2169,0.109845
1,development,59670,41960,6854,0.114865
2,test,19927,14005,2291,0.114970


In [5]:
display(pd.read_csv(ROOT / 'artifacts/tables/data_dictionary.csv'))

,column,role,type,description,missing_token,source
0,encounter_id,identifier,categorical,"Unique encounter ID; audit/join only, never a ...",?,https://archive.ics.uci.edu/dataset/296/diabet...
1,patient_nbr,identifier,categorical,Patient grouping ID; used to prevent split ove...,?,https://archive.ics.uci.edu/dataset/296/diabet...
2,race,candidate input,categorical,Recorded race category; missing values remain ...,?,https://archive.ics.uci.edu/dataset/296/diabet...
3,gender,candidate input,categorical,Recorded gender category; Unknown/Invalid reta...,?,https://archive.ics.uci.edu/dataset/296/diabet...
4,age,candidate input,categorical,"Age in ten-year bands; categorical, not exact ...",?,https://archive.ics.uci.edu/dataset/296/diabet...
5,weight,candidate input,categorical,Weight band; subject to training-only high-mis...,?,https://archive.ics.uci.edu/dataset/296/diabet...
6,admission_type_id,candidate input,categorical,Categorical admission type; see IDS_mapping.csv.,?,https://archive.ics.uci.edu/dataset/296/diabet...
7,discharge_disposition_id,candidate input,categorical,Categorical discharge destination; used for co...,?,https://archive.ics.uci.edu/dataset/296/diabet...
8,admission_source_id,candidate input,categorical,Categorical admission source; see IDS_mapping....,?,https://archive.ics.uci.edu/dataset/296/diabet...
9,time_in_hospital,candidate input,integer,Length of the completed hospital stay in days.,?,https://archive.ics.uci.edu/dataset/296/diabet...


In [6]:
from readmission.data import load_cohort
cohort = load_cohort()
assert cohort.groupby('patient_nbr')['split'].nunique().max() == 1
assert cohort['target'].sum() == 11314
print('Verified patient-disjoint splits and cohort labels.')

Verified patient-disjoint splits and cohort labels.


## Team interpretation prompts

Explain why patient-level splitting matters, what the positive label actually captures, and why 11.4% refers to encounters.

Write your own report discussion after checking the evidence.